In [0]:
import os
import sys


SRC_PATH = os.path.abspath(
    "../src"
)

if SRC_PATH not in sys.path:
    sys.path.insert(
        0,
        SRC_PATH
    )

print(
    f"Project source path: {SRC_PATH}"
)

from sample_assignment.ingestion import RawDataIngestion
from sample_assignment.data_quality import DataQualityProcessor
from sample_assignment.enrichment import EnrichmentProcessor
from sample_assignment.aggregation import AggregationProcessor



Project source path: /Workspace/Users/manisha.tech.dey@outlook.com/sales_project_simplified_new/src


In [0]:
CATALOG_NAME = "retail_sales"

VOLUME_PATH = (
    "/Volumes/retail_sales/raw/"
)

print("=" * 70)
print("RETAIL SALES RAW INGESTION")
print("=" * 70)
print(f"Catalog     : {CATALOG_NAME}")
print(f"Volume path : {VOLUME_PATH}")
print("=" * 70)

RETAIL SALES RAW INGESTION
Catalog     : retail_sales
Volume path : /Volumes/retail_sales/raw/


In [0]:
ingestion_pipeline = RawDataIngestion(
    spark=spark,
    volume_path=VOLUME_PATH,
    catalog=CATALOG_NAME,
)

ingestion_result = (
    ingestion_pipeline.run()
)

Ingesting source : products
Target table     : retail_sales.raw.products
products: 1851 rows written to retail_sales.raw.products
Ingesting source : orders
Target table     : retail_sales.raw.orders
orders: 9994 rows written to retail_sales.raw.orders


In [0]:
print("Starting Data Quality...")

data_quality_result = DataQualityProcessor(
    spark=spark,
    catalog=CATALOG_NAME,
).run()

if data_quality_result["status"] != "SUCCESS":
    raise RuntimeError("Data Quality failed.")

In [0]:
print("Starting Enrichment...")

enrichment_result = EnrichmentProcessor(
    spark=spark,
    catalog=CATALOG_NAME,
).run()

if enrichment_result["status"] != "SUCCESS":
    raise RuntimeError("Enrichment failed.")

In [0]:
print("Starting Aggregation...")

aggregation_result = AggregationProcessor(
    spark=spark,
    catalog=CATALOG_NAME,
).run()

if aggregation_result["status"] != "SUCCESS":
    raise RuntimeError("Aggregation failed.")

In [0]:
pipeline_summary = [
    ("ingestion", "SUCCESS"),
    ("data_quality", data_quality_result["status"]),
    ("enrichment", enrichment_result["status"]),
    ("aggregation", aggregation_result["status"]),
]

display(
    spark.createDataFrame(
        pipeline_summary,
        ["pipeline_stage", "status"],
    )
)

print("RETAIL SALES PIPELINE COMPLETED SUCCESSFULLY")